# Quantum K-Means — Detailed Notes (Session 19)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller

> **Purpose.** These notes turn the slide bullets into a stand-alone reference for *quantum-enhanced K-means*: how to encode vectors, estimate distances/inner products with quantum circuits, build a hybrid Lloyd’s loop, and evaluate clustering quality on NISQ-era devices (or simulators). We emphasize realistic, hybrid pipelines and modern Qiskit primitives.

---

## Session road-map
1. Recap: Quantum Bayes → why distances now  
2. Classical K-means recap & where quantum fits  
3. Encodings: amplitude vs. angle (and normalization)  
4. Quantum distance/similarity estimation  
   - Swap test (overlap)  
   - Hadamard test (inner product sign/phase)  
   - From similarity to Euclidean distance  
   - (Optional) Amplitude Estimation for precision  
5. Hybrid quantum K-means loops (two variants)  
6. Qiskit implementations (primitives-based)  
7. Performance & scalability analysis  
8. Evaluation metrics and experimental protocol  
9. Practical tips & pitfalls  
10. Mini-exercises (with brief solutions)

---

## 0) Recap → From probabilities to distances
- Quantum Bayes: used **amplitude estimation** to estimate probabilities faster (ideally).  
- For clustering, the central operation is **assigning points to nearest centroids**, which needs **distances/inner products** in high dimensions.  
- Quantum circuits can estimate **overlaps**/inner products between *encoded* states and may reduce sample complexity for similarity estimation; in practice we use **hybrid loops**.

---

## 1) Classical K-means (Lloyd’s algorithm)
- Iterate until convergence:  
  1) **Assignment:** for each point $x_i$, assign to cluster $c=\arg\min_k \|x_i-\mu_k\|^2$.  
  2) **Update:** recompute centroids $\mu_k$ as means of assigned points.
- Cost driver: computing $\|x_i-\mu_k\|$ for many $(i,k)$ pairs.

**Quantum insertion point:** use circuits to estimate **$\langle x | \mu_k \rangle$** or **$|\langle x | \mu_k \rangle|^2$** (given encodings), then derive distances.

---

## 2) Data encodings

### 2.1 Amplitude encoding (compact but deeper)
Encode $x\in\mathbb{R}^d$ into $|x\rangle=\sum_{j=0}^{d-1} \frac{x_j}{\|x\|}\,|j\rangle$ using $n=\lceil\log_2 d\rceil$ qubits.  
- **Pros:** logarithmic qubits in $d$.  
- **Cons:** **state preparation** can be costly and deep for arbitrary $x$; sensitive to noise.

### 2.2 Angle (feature) encoding (shallow, more qubits)
Apply rotations per feature, e.g. $R_y(\alpha x_j)$ on qubit $j$ and interleave entanglers (ZZ feature map).  
- **Pros:** shallow, hardware-friendly.  
- **Cons:** uses ~$d$ qubits; gives a *kernel* implicitly rather than a strict amplitude-overlap.

> **Guidance:** For toy problems and NISQ: start with **angle/ZZ** encoding. Use amplitude encoding on simulator or when you can afford robust state prep.

**Normalization:** Always scale inputs (e.g., standardize and map to $[-\pi,\pi]$ for angles; L2-normalize for amplitude encoding).

---

## 3) Quantum distance & similarity estimation

### 3.1 Overlap via **swap test** (amplitude encoding)
Given $|x\rangle,|y\rangle$, the swap test returns  
$$
\Pr(\text{ancilla}=0) = \tfrac{1}{2}\big(1+|\langle x|y\rangle|^2\big).
$$  
If vectors are L2-normalized, **cosine similarity** $s= \langle x|y\rangle$ and  
$$
\|x-y\|^2 = 2(1-s).
$$
- Circuit: H on ancilla → CSWAP(ancilla, reg-x, reg-y) → H on ancilla → measure.

### 3.2 Inner product via **Hadamard test** (phase-based)
Prepare $|x\rangle$ and implement a **unitary** $U$ that maps $|x\rangle\mapsto|y\rangle$ (or encodes $y$ as a phase kick). The Hadamard test estimates **$\operatorname{Re}\langle x|U|x\rangle$** (and with phase shifts, the imaginary part), giving signed inner products when accessible. More tailored but requires constructing $U$.

### 3.3 Angle-encoded similarity (quantum kernel)
With angle/ZZ feature maps $U_\phi(x)$, define a **quantum kernel**  
$$
K(x,y)=|\langle 0|U_\phi^\dagger(x)U_\phi(y)|0\rangle|^2.
$$
Use **kernel K-means** (or spectral clustering) with $K$ instead of Euclidean distances—well-suited to NISQ.

### 3.4 Precision: shots vs **Amplitude Estimation (AE)**
- Plain sampling: std. error $O(1/\sqrt{S})$ with $S$ shots.  
- AE/Iterative AE: $O(1/\epsilon)$ oracle calls for additive error $\epsilon$ (asymptotically), but deeper circuits/oracle costs may offset gains on NISQ. Prefer **Iterative AE** if you try AE.

---

## 4) Hybrid quantum K-means variants

### Variant A — **Distances with swap/Hadamard tests**
1) Classically initialise centroids $\{\mu_k\}$.  
2) For each point $x_i$, **quantum-estimate** similarity $s_{ik} \approx \langle x_i|\mu_k\rangle$.  
3) Assign $x_i \to \arg\min_k 2(1-s_{ik})$.  
4) Update $\mu_k$ **classically** (mean of assigned points).  
5) Repeat.

### Variant B — **Kernel K-means with quantum kernel**
1) Choose feature map $U_\phi$; estimate $K(x_i,x_j)$ quantumly (possibly on-the-fly).  
2) Run **kernel K-means** or **spectral clustering** with kernel $K$.  
3) (Optional) Precompute kernel matrix for small $N$ and cache.

> **Practice:** B is often more NISQ-friendly: shallow circuits, no explicit centroids in quantum memory, good with small datasets.

---

## 5) Qiskit implementations (primitives-based)

> Prefer **primitives** (`Sampler`, `Estimator`). The older Opflow APIs are deprecated.

### 5.1 Swap test helper (amplitude-encoded vectors)
```python
# pip install qiskit qiskit-aer numpy
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler

def stateprep_from_vector(vec):
    """Return a circuit that prepares |vec> on n qubits (len(vec)=2^n)."""
    # For real, normalized vec; for general data, use a proper state-prep routine.
    n = int(np.log2(len(vec)))
    qc = QuantumCircuit(n)
    qc.initialize(vec / np.linalg.norm(vec), qc.qubits)
    return qc

def swap_test_overlap(x_vec, y_vec, shots=4096):
    nx = int(np.log2(len(x_vec))); ny = int(np.log2(len(y_vec)))
    assert nx == ny, "Dimension mismatch"
    n = nx
    prep_x = stateprep_from_vector(x_vec)
    prep_y = stateprep_from_vector(y_vec)

    anc = 1; reg = n
    qc = QuantumCircuit(1 + 2*n, name='swap-test')
    # load states
    qc.compose(prep_x, qubits=range(1, 1+n), inplace=True)
    qc.compose(prep_y, qubits=range(1+n, 1+2*n), inplace=True)
    # swap test
    qc.h(0)
    for q in range(n):
        qc.cswap(0, 1+q, 1+n+q)
    qc.h(0)
    # measure ancilla only via Sampler (no classical reg needed)
    samp = Sampler()
    res = samp.run([qc], shots=shots).result()
    probs = res.quasi_dists[0]
    p0 = probs.get(0, 0.0)  # ancilla=|0...> key 0
    overlap2 = max(0.0, min(1.0, 2*p0 - 1))  # = |<x|y>|^2
    return overlap2
```

### 5.2 Quantum kernel with angle/ZZ feature map
```python
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer.primitives import Sampler
from qiskit import QuantumCircuit

def kernel_entry(x, y, feature_map, shots=2048):
    """Compute K(x,y)=|<0|U†(x)U(y)|0>|^2 with Sampler."""
    n = feature_map.num_qubits
    Ux = feature_map.assign_parameters(dict(zip(feature_map.parameters, x)))
    Uy = feature_map.assign_parameters(dict(zip(feature_map.parameters, y)))
    qc = QuantumCircuit(n)
    qc.compose(Uy, inplace=True)
    qc.compose(Ux.inverse(), inplace=True)
    samp = Sampler()
    res = samp.run([qc], shots=shots).result()
    p0 = res.quasi_dists[0].get(0, 0.0)  # probability of |00...0>
    return p0
```

### 5.3 Hybrid loop sketch (kernel K-means)
```python
import numpy as np
from sklearn.cluster import KMeans

def quantum_kernel_matrix(X, feature_map, shots=2048):
    N, d = X.shape
    Kmat = np.zeros((N, N))
    for i in range(N):
        Kmat[i, i] = 1.0
        for j in range(i+1, N):
            kij = kernel_entry(X[i], X[j], feature_map, shots=shots)
            Kmat[i, j] = Kmat[j, i] = kij
    return Kmat

def kernel_kmeans_from_K(Kmat, K):
    # Simple wrapper: run KMeans on top eigen-space (spectral trick) or
    # use precomputed-kernel variants (libraries vary). Here: spectral sketch.
    from sklearn.manifold import SpectralEmbedding
    Z = SpectralEmbedding(n_components=K, affinity='precomputed').fit_transform(Kmat)
    labels = KMeans(n_clusters=K, n_init=10, random_state=0).fit_predict(Z)
    return labels

# Example usage (small 2D data with ZZ map)
from qiskit.circuit.library import ZZFeatureMap
feature_map = ZZFeatureMap(feature_dimension=2, reps=1)
# X: shape [N,2]; scale to [-pi, pi] before calling
```

> **Tip:** Cache `U(x)` circuits and reuse; batch Sampler calls to amortise latency.

---

## 6) Performance & scalability

Let $N$ = points, $K$ = clusters, $d$ = data dim, $S$ = shots.

- **Classical Lloyd:** assignment $O(NK d)$ per iteration.  
- **Quantum distances (amplitude encoding):** overlap per $(i,k)$ = one swap test $\Rightarrow O(1)$ circuit depth plus state-prep (**dominant**). If state prep is black-box, distance precision $\epsilon$ costs $O(1/\epsilon^2)$ shots (or $O(1/\epsilon)$ with AE).  
- **Quantum kernel:** precompute $K$ for $N$ points → $O(N^2)$ circuit evaluations; works for small $N$ (lab scale). For larger $N$, compute on-the-fly batches.

**When advantage is plausible (theory):**
- Very high-dimensional data where amplitude encoding is efficient (oracle access) and AE is usable.  
- Kernel regimes where classical kernels struggle but quantum feature maps capture structure with shallow circuits.

**NISQ reality:** state prep + noise often dominate. **Hybrid** methods + small $N$ are the practical target.

---

## 7) Evaluation of clustering quality

- **Internal metrics:** Silhouette score, Davies–Bouldin (no labels required).  
- **External metrics (with labels):** Adjusted Rand Index (ARI), Normalized Mutual Information (NMI).  
- **Protocol:**  
  1) Standardize features; for angle encoding, map to $[-\pi,\pi]$.  
  2) Fix a shot budget and report mean±std across seeds.  
  3) Compare to baselines: K-means (Euclidean), kernel K-means (RBF), spectral clustering.  
  4) Ablations: shots (256→4096), feature-map reps (1→2), noise vs ideal simulator.

---

## 8) Practical tips & pitfalls

- **State prep cost:** the hidden dragon. Use **feature maps** (angle/ZZ) unless amplitude encoding is cheap.  
- **Normalization:** L2-normalize for swap test; standardize for angle maps.  
- **Transpilation:** set optimization level 2–3; bind parameters instead of recompiling; map to high-fidelity qubits.  
- **Shot budgeting:** start low (256–512) for coarse assignment; increase near convergence.  
- **Caching:** pre-bind feature maps for unique $x$; reuse inverse $U(x)^\dagger$.  
- **Noise mitigation:** readout calibration; (optionally) zero-noise extrapolation for kernel entries.  
- **Robustness:** stop criteria tolerant to noisy distances (e.g., assignment stabilizes for 2 consecutive iterations).  
- **Fair baselines:** match parameter count/depth with small classical kernels.

---

## 9) Mini-exercises (answers below)

1. **From overlap to distance:** For normalized $x,y$, show $\|x-y\|^2 = 2(1-|\langle x|y\rangle|)$ for **cosine similarity** vs $2(1-|\langle x|y\rangle|^2)$ for **swap-test overlap**; when do they coincide?  
2. **Swap test math:** Derive $\Pr(\text{anc}=0)=\tfrac12(1+|\langle x|y\rangle|^2)$.  
3. **Shot error:** If the ancilla outcome has variance $p(1-p)$, what shot count $S$ yields std. error $\le 0.02$ on the overlap estimate when $p\approx 0.75$?  
4. **Kernel K-means objective:** Given kernel matrix $K$, write the within-cluster objective in terms of $K$ and assignment indicators.  
5. **Complexity sanity-check:** For $N=200$, $K=3$, one iteration of kernel K-means with precomputed $K$: how many quantum evaluations if you compute the full matrix once (no symmetry tricks) vs with symmetry?

---

## 10) Summary (Session 19)
- Quantum K-means = classical Lloyd’s algorithm with a **quantum distance/similarity oracle** or **quantum kernel**.  
- On NISQ, **kernel variants** with shallow feature maps are most practical; amplitude encoding + swap/AE is instructive on simulators.  
- Evaluate with proper baselines, report shot/latency costs, and use caching/mitigation.  
- Realistic wins: **small $N$**, **non-linear structure**, and **hybrid pipelines**.

---

## 11) Looking ahead
- **Next Session:** Quantum GANs — generators/discriminators with PQCs and adversarial training loops.  
- **Homework 5 (Quantum K-means):**  
  1) Build a ZZ-feature quantum kernel on the moons dataset; run kernel K-means and report ARI vs shots.  
  2) Compare to RBF kernel K-means (tuned $\gamma$).  
  3) (Bonus) Implement swap-test distances on 4-dim amplitude-encoded data; compare silhouette to classical Euclidean.

---

## Appendix — mini-exercise solutions (sketch)

1. **Overlap→distance.** For L2-normalized vectors, cosine similarity $s=\langle x,y\rangle$. Euclidean: $\|x-y\|^2 = \|x\|^2+\|y\|^2-2\langle x,y\rangle = 2(1-s)$. Swap test yields $|s|^2$. They coincide when $s\ge 0$ *and* $s\approx |s|^2$ only for $s\in\{0,1\}$; otherwise you must convert appropriately (prefer direct inner product if sign matters).  
2. **Swap test.** Starting state $\tfrac{1}{\sqrt2}(|0\rangle|x\rangle|y\rangle+|1\rangle|y\rangle|x\rangle)$ after first H+CSWAP. Final H yields ancilla |0⟩ amplitude $\tfrac{1}{2}(\,|x\rangle|y\rangle+|y\rangle|x\rangle\,)$. Probability is $\tfrac{1}{2}(1+|\langle x|y\rangle|^2)$.  
3. **Shots.** Std. error of $p$ is $\sqrt{p(1-p)/S}$. With $p\approx0.75$: $\sqrt{0.75\cdot0.25/S}\le0.02 \Rightarrow S \gtrsim 0.75\cdot0.25/0.0004 \approx 469$. Round up to ~512 shots per estimate.  
4. **Kernel K-means objective.** For cluster $C_k$ with indicator $u_{ik}\in\{0,1\}$, size $n_k$:  
   $J = \sum_{k}\sum_{i\in C_k}\|\phi(x_i)-m_k\|^2$ with $m_k=\frac1{n_k}\sum_{j\in C_k}\phi(x_j)$. Expanding via kernel trick:  
   $J = \sum_{k}\left[\sum_{i\in C_k}K_{ii} - \frac{2}{n_k}\sum_{i,j\in C_k}K_{ij} + \frac{1}{n_k^2}\sum_{i,j\in C_k}K_{ij}\right].$  
   Since $K_{ii}=1$ for many quantum kernels, this simplifies.  
5. **Evaluations.** Full matrix once: $N^2=40{,}000$ entries (or $N(N-1)/2\approx 19{,}900$ with symmetry + diagonals known as 1). Use symmetry to **halve** quantum evaluations.

